# NNCP run traces

Reads `step ... bytes ... bpb ... window_bpb ... elapsed ...s` lines out of
`records/*-nncp.log` and plots the bpb trajectory for each run.

- `total_bpb` is the cumulative compressed-bits-per-input-byte at each
  reporting step (the actual artifact ratio so far).
- `window_bpb` is the bits-per-byte over just the most recent 1024-byte
  window, so it tracks how well the model is predicting in the present
  rather than amortizing the cold-start.
- The dashed lines mark the classical compression floors on the same data
  stream (gzip, xz). NNCP needs `total_bpb` < xz to claim a real win.

In [ ]:
from __future__ import annotations

import json
import re
from pathlib import Path

import matplotlib.pyplot as plt

ROOT = Path('.').resolve()
if ROOT.name == 'records':
    ROOT = ROOT.parent
RECORDS = ROOT / 'records'
STEP_RE = re.compile(
    r'step\s+(?P<step>\d+)\s+bytes\s+(?P<bytes>\d+)\s+bits\s+(?P<bits>\d+)'
    r'\s+bpb\s+(?P<bpb>\S+)\s+window_bpb\s+(?P<win>\S+)\s+elapsed\s+(?P<sec>\S+)s'
)


def parse_log(path: Path) -> list[dict[str, float]]:
    rows: list[dict[str, float]] = []
    for line in path.read_text(encoding='utf-8', errors='replace').splitlines():
        m = STEP_RE.search(line)
        if not m:
            continue
        rows.append(
            {
                'bytes': int(m['bytes']),
                'bits': int(m['bits']),
                'bpb': float(m['bpb']),
                'window_bpb': float(m['win']),
                'elapsed_s': float(m['sec']),
            }
        )
    return rows


logs = sorted(RECORDS.glob('*-nncp.log'))
logs

In [ ]:
# Classical floors, by dataset prefix in the log filename.
CLASSICAL_FLOORS = {
    'monaco': {'gzip': 5.396, 'xz': 4.614},
    'paris': {'gzip': 4.627, 'xz': 3.750},
    'england': {'gzip': 4.839, 'xz': 4.069},
}


def dataset_for(name: str) -> str:
    n = name.lower()
    if 'paris' in n:
        return 'paris'
    if 'england' in n:
        return 'england'
    return 'monaco'


In [ ]:
fig, axes = plt.subplots(len(logs), 1, figsize=(11, 3.2 * len(logs)), sharex=False)
if len(logs) == 1:
    axes = [axes]

for ax, log in zip(axes, logs):
    rows = parse_log(log)
    if not rows:
        ax.set_title(f'{log.name}  (no step lines)')
        continue
    xs = [r['bytes'] for r in rows]
    ax.plot(xs, [r['bpb'] for r in rows], label='total bpb', linewidth=1.5)
    ax.plot(xs, [r['window_bpb'] for r in rows], label='window bpb', alpha=0.4)
    floors = CLASSICAL_FLOORS.get(dataset_for(log.name), {})
    for label, val in floors.items():
        ax.axhline(val, linestyle='--', linewidth=0.8, alpha=0.7, label=f'{label} floor')
    ax.set_title(log.name)
    ax.set_xlabel('bytes processed')
    ax.set_ylabel('bpb')
    ax.set_ylim(3.0, 8.2)
    ax.legend(loc='upper right', fontsize=8)
    ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

In [ ]:
# Overlay all runs on the same axes, total bpb only, for comparison.
plt.figure(figsize=(11, 5))
for log in logs:
    rows = parse_log(log)
    if not rows:
        continue
    xs = [r['bytes'] for r in rows]
    ys = [r['bpb'] for r in rows]
    plt.plot(xs, ys, label=log.stem)
for ds, floors in CLASSICAL_FLOORS.items():
    for label, val in floors.items():
        plt.axhline(val, linestyle='--', linewidth=0.7, alpha=0.5, label=f'{ds} {label}')
plt.xscale('log')
plt.xlabel('bytes processed (log scale)')
plt.ylabel('total bpb')
plt.title('NNCP runs: total bpb trajectory')
plt.ylim(3.0, 8.2)
plt.grid(True, alpha=0.3)
plt.legend(loc='upper right', fontsize=7)
plt.tight_layout()
plt.show()